# Laboratorio 04.2 — Evaluación con LLM como juez

Diseñaremos una rúbrica legible por un juez LLM, validaremos su respuesta con Pydantic y compararemos la valoración opcional con una referencia humana.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Escribir una rúbrica con criterios observables y escala definida.
- Construir una evaluación pareada con respuesta de referencia y candidata.
- Validar el resultado del juez y registrar incumplimientos del esquema.
- Identificar sesgos, desacuerdos y límites de usar un modelo para juzgar otro.

## 2. Conceptos

Un LLM como juez asigna puntuaciones según una rúbrica y evidencia. Puede escalar evaluación de respuestas abiertas, pero introduce sesgo por posición, estilo, longitud y familiaridad con su propio proveedor. La evaluación debe calibrarse con juicios humanos, ejemplos pareados y análisis de desacuerdos. La salida del juez también es generada y necesita validación.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Pregunta y referencia] --> C[Prompt de rubrica]
    B[Respuesta candidata] --> C
    C --> D[Juez LLM opcional]
    D --> E[JSON de evaluacion]
    E --> F[Validacion Pydantic]
    G[Juicio humano] --> H[Analisis de acuerdo]
    F --> H
```

## 4. Preparación del entorno

Usaremos pandas, Pydantic y el adaptador común de modelos. El juez permanece apagado; el ejemplo local solo calcula una línea base léxica y no se presenta como juicio semántico.

In [1]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import sys

raices_candidatas = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
raiz_curso = next(
    (
        candidato
        for raiz in raices_candidatas
        for candidato in (raiz / 'genai', raiz)
        if (candidato / 'src' / 'genai').is_dir()
    ),
    None,
)
if raiz_curso is None:
    raise FileNotFoundError('No se encontró el proyecto genAI con src/genai.')

sys.path.insert(0, str(raiz_curso / 'src'))

In [2]:
import json
from time import perf_counter

import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from genai.environment import preparar_entorno
from genai.models import get_llm

raiz_curso = preparar_entorno()

## 5. Definición del problema

Evaluamos respuestas a una pregunta sobre la política de devolución. El conjunto es sintético y tiene dos candidatas con valoraciones humanas de referencia para ilustrar desacuerdos; no se usará para afirmar la calidad general de un juez.

## 6. Línea base

La coincidencia de palabras con la respuesta de referencia es una línea base sencilla. No reconoce paráfrasis ni verifica entailment, por lo que la usaremos como contraste, no como evaluador final.

In [3]:
pregunta = '¿Cuál es el plazo para devolver un artículo?'
referencia = 'Se aceptan devoluciones durante 30 días desde la entrega.'
candidatas = [
    {'id': 'A', 'texto': 'Tienes 30 días desde que recibes el artículo.', 'puntuacion_humana': 5},
    {'id': 'B', 'texto': 'El plazo es de 300 días y empieza cuando haces la compra.', 'puntuacion_humana': 1},
]

def puntuacion_lexica(texto: str, respuesta_referencia: str) -> float:
    """Calcula una fraccion de palabras de referencia presentes en el texto."""
    palabras = set(respuesta_referencia.casefold().split())
    if not palabras:
        return 0.0
    return len(palabras & set(texto.casefold().split())) / len(palabras)

base_juez = pd.DataFrame([
    {
        'candidata': candidata['id'],
        'puntuacion_humana': candidata['puntuacion_humana'],
        'coincidencia_lexica': puntuacion_lexica(candidata['texto'], referencia),
    }
    for candidata in candidatas
])
display(base_juez)

,candidata,puntuacion_humana,coincidencia_lexica
0,A,5,0.333333
1,B,1,0.222222


## 7. Implementación

La rúbrica separa exactitud factual, relevancia y claridad. El esquema limita la puntuación a 1–5 y exige una justificación concisa; una respuesta fuera de contrato no se incorpora como puntuación válida.

In [4]:
class EvaluacionJuez(BaseModel):
    model_config = ConfigDict(extra='forbid')

    puntuacion: int = Field(ge=1, le=5)
    justificacion: str = Field(min_length=12, max_length=300)
    problemas: list[str] = Field(default_factory=list, max_length=5)


def construir_prompt_juez(pregunta: str, respuesta_referencia: str, respuesta: str) -> str:
    """Construye una solicitud de juicio con escala y criterios explícitos."""
    return f'''Evalua la respuesta candidata con la rubrica.
Puntua de 1 a 5: 1 contiene errores graves; 3 es parcialmente correcta; 5 responde correctamente y de forma pertinente.
Considera exactitud factual, relevancia y claridad. No otorgues puntos por longitud o estilo por si solos.
Devuelve un objeto JSON con puntuacion, justificacion y problemas.

Pregunta: {pregunta}
Referencia: {respuesta_referencia}
Respuesta candidata: {respuesta}
'''

prompt_juez = construir_prompt_juez(pregunta, referencia, candidatas[0]['texto'])
print(prompt_juez)

Evalua la respuesta candidata con la rubrica.
Puntua de 1 a 5: 1 contiene errores graves; 3 es parcialmente correcta; 5 responde correctamente y de forma pertinente.
Considera exactitud factual, relevancia y claridad. No otorgues puntos por longitud o estilo por si solos.
Devuelve un objeto JSON con puntuacion, justificacion y problemas.

Pregunta: ¿Cuál es el plazo para devolver un artículo?
Referencia: Se aceptan devoluciones durante 30 días desde la entrega.
Respuesta candidata: Tienes 30 días desde que recibes el artículo.



## 8. Experimentos

Las candidatas comparten pregunta y referencia. Si se activa el juez, cada candidata produce como máximo una solicitud: dos llamadas en total, cero por defecto. OpenAI y Anthropic pueden tener costo; Ollama ofrece una alternativa local si el servicio está disponible.

In [5]:
EVALUAR_CON_LLM = False
PROVEEDOR_JUEZ = 'ollama'
llamadas_maximas = len(candidatas)
print(f'Llamadas maximas previstas: {llamadas_maximas}; llamadas por defecto: 0')
filas_juez = []
if EVALUAR_CON_LLM:
    modelo_juez = get_llm(PROVEEDOR_JUEZ)
    for candidata in candidatas:
        inicio = perf_counter()
        try:
            respuesta_modelo = modelo_juez.invoke(
                construir_prompt_juez(pregunta, referencia, candidata['texto'])
            )
            evaluacion = EvaluacionJuez.model_validate_json(str(respuesta_modelo.content))
            filas_juez.append({
                'candidata': candidata['id'],
                'puntuacion_juez': evaluacion.puntuacion,
                'schema_valid': True,
                'latencia_segundos': perf_counter() - inicio,
                'tipo_error': None,
            })
        except (ValidationError, ValueError) as error:
            filas_juez.append({
                'candidata': candidata['id'],
                'puntuacion_juez': None,
                'schema_valid': False,
                'latencia_segundos': perf_counter() - inicio,
                'tipo_error': type(error).__name__,
            })

tabla_juez = pd.DataFrame(filas_juez)
display(tabla_juez)

Llamadas maximas previstas: 2; llamadas por defecto: 0


""


## 9. Evaluación

Comparamos puntuaciones del juez con juicios humanos de referencia y medimos cumplimiento de esquema. Con cero llamadas la tabla queda vacía deliberadamente. Incluso con respuestas del juez, dos ejemplos no bastan para estimar acuerdo robusto.

In [6]:
if not tabla_juez.empty:
    referencia_humana = pd.DataFrame(candidatas)[['id', 'puntuacion_humana']].rename(
        columns={'id': 'candidata'}
    )
    comparacion_juez = tabla_juez.merge(referencia_humana, on='candidata')
    acuerdo_ordinal = comparacion_juez['puntuacion_juez'].corr(
        comparacion_juez['puntuacion_humana'], method='spearman'
    )
    tasa_schema = float(tabla_juez['schema_valid'].mean())
else:
    comparacion_juez = pd.DataFrame()
    acuerdo_ordinal = None
    tasa_schema = None
metricas_juez = pd.DataFrame([{
    'llamadas_realizadas': len(tabla_juez),
    'correlacion_spearman': acuerdo_ordinal,
    'schema_valid': tasa_schema,
    'costo_estimado': None,
}])
display(comparacion_juez)
display(metricas_juez)
if not tabla_juez.empty:
    tabla_juez.to_csv(raiz_curso / 'outputs' / '04_02_judge_results.csv', index=False)

""


,llamadas_realizadas,correlacion_spearman,schema_valid,costo_estimado
0,0,None,None,None


## 10. Discusión

Un juez puede favorecer respuestas largas, cierto estilo o su propio proveedor, y puede fallar al seguir la rúbrica. Calibra criterios con evaluadores humanos, oculta la identidad del modelo cuando sea posible y audita discrepancias. No uses un juez LLM como única barrera de seguridad ni como verdad absoluta.

## 11. Ejercicios

1. Reescribe la rúbrica con anclajes explícitos para cada puntuación.
2. Añade una respuesta correcta pero más breve y analiza el sesgo de longitud.
3. Intercambia el orden de dos candidatas y mide si cambia el juicio.
4. Compara un juez local y uno remoto con la misma rúbrica y presupuesto.

## 12. Desafío

Diseña una calibración con al menos 30 pares evaluados por personas y por dos jueces. Estima acuerdo ordinal, desglosa desacuerdos por tipo de error y propón una regla de abstención cuando los jueces discrepen.

## 13. Ideas clave

- Un juez necesita una rúbrica explícita y una salida validable.
- La evaluación automática debe calibrarse con referencias humanas.
- Posición, estilo y proveedor pueden introducir sesgo.
- El juez es otro componente probabilístico que también se evalúa.